# Speculative Decoding Training via Kubeflow Trainer SDK — Red Hat OpenShift AI Example

This notebook demonstrates how to train a custom **Eagle3 draft model** for speculative decoding
using the `SpeculativeDecodingTrainer` from the Kubeflow SDK on Red Hat OpenShift AI.

## What is Speculative Decoding?

Large language models generate tokens one at a time during inference, and each token requires
reading the entire model from GPU memory — making inference **memory-bound**. The GPU's compute
units sit idle most of the time, waiting for data to arrive from memory.

Speculative decoding exploits this: verifying multiple tokens at once costs almost the same as
generating one, because the expensive memory read is identical and the extra compute is tiny.
A small, fast **draft model** (~1 GB) guesses the next several tokens, then the large **verifier
model** checks all guesses in a single forward pass. Correct guesses become output; at the first
rejected guess, the verifier's own token is used. The output is mathematically identical to
normal decoding — no quality loss.

**Eagle3** is a draft model architecture that reads hidden states from four intermediate layers
of the verifier (not just the final logits), giving it richer context for more accurate
predictions. After fine-tuning a model, the pre-built draft model's acceptance rate drops — so
you need a **custom draft model** trained for your specific verifier to achieve meaningful speedup.

## What This Notebook Covers

This notebook demonstrates all four training modes of `SpeculativeDecodingTrainer`:

1. **DATA_ONLY** — Extract hidden states from the verifier using a managed vLLM sidecar
2. **TRAIN_ONLY** — Train the draft model from pre-extracted hidden states (uses output from step 1)
3. **OFFLINE** — Extract + train using a user-managed external vLLM endpoint
4. **ONLINE** — Fully managed end-to-end extraction and training

## Supported Datasets

The SDK provides three built-in dataset names that can be used directly with `dataset_name`:

| Name | Description |
|------|-------------|
| `ultrachat` | Multi-turn conversational dataset |
| `magpie` | Magpie-format conversation dataset |
| `gsm8k` | Grade school math word problems |

You can also pass a HuggingFace dataset ID, a PVC URI (`pvc://<name>/<path>`), or a local
`.json`/`.jsonl` file path as `dataset_name`.

## Hardware Requirements

| Component | GPU | CPU | Memory |
|-----------|-----|-----|--------|
| Training container | 2× NVIDIA L40S / A100 | 4 cores | 64Gi |
| vLLM sidecar | 1× NVIDIA L40S / A100 | 2 cores | 64Gi |

## Setup

Install the Kubeflow SDK and import required dependencies.

In [ ]:
# Install the Kubeflow SDK from the OpenDataHub fork (includes SpeculativeDecodingTrainer)
!pip install --no-cache-dir --force-reinstall --no-deps git+https://github.com/opendatahub-io/kubeflow-sdk.git@main

# Structured logging library (dependency for SDK progress tracking)
!pip install structlog

# Kubeflow Trainer API — provides TrainerClient, KubernetesBackendConfig, and job management
!pip install --no-cache-dir --force-reinstall --index-url https://pypi.org/simple kubeflow-trainer-api==2.3.0

In [ ]:
import os

# Kubernetes Python client — used to configure API server auth and create the API client
from kubernetes import client
import kubeflow

# Backend config tells the TrainerClient how to connect to the cluster
from kubeflow.common.types import KubernetesBackendConfig

# TrainerClient is the main entry point for submitting, monitoring, and deleting TrainJobs
from kubeflow.trainer import TrainerClient

# Name option lets you assign an explicit name to a TrainJob (otherwise auto-generated)
from kubeflow.trainer.options.common import Name

# Red Hat OpenShift AI extensions for speculative decoding training
from kubeflow.trainer.rhai import (
    SpeculativeDecodingTrainer,  # High-level trainer that wraps all four modes
    SpeculatorConfig,            # Fine-grained config: layer IDs, architecture, scheduler, etc.
    SpeculatorMode,              # Enum: DATA_ONLY, TRAIN_ONLY, OFFLINE, ONLINE
    SpeculatorType,              # Enum: EAGLE3 (currently the only supported type)
)

print(f"Kubeflow SDK version: {kubeflow.__version__}")
print("All imports successful")

In [ ]:
# Verify the SDK loaded correctly by inspecting the available enum values and defaults.
# This confirms that SpeculatorMode, SpeculatorType, and SpeculatorConfig are importable
# and behave as expected before proceeding to cluster authentication.
print(f"Modes: {[m.value for m in SpeculatorMode]}")
print(f"Types: {[t.value for t in SpeculatorType]}")
print(f"Config defaults: {SpeculatorConfig()}")
print("SDK ready")

## Authenticate to your OpenShift Cluster

Provide your OpenShift API server URL, authentication token, and HuggingFace token.
Update `PVC_NAME` to match the name of your shared RWX PersistentVolumeClaim.

In [ ]:
# ============================================================================
# CLUSTER AUTHENTICATION
# ============================================================================
# Replace these with your OpenShift cluster API server URL and bearer token.
# In OpenShift AI workbenches, these may be available as environment variables
# (OPENSHIFT_API_URL, NOTEBOOK_USER_TOKEN) — but for clarity we set them explicitly.
api_server = "<REPLACE WITH OPENSHIFT SERVER>"
token = "<REPLACE WITH OPENSHIFT TOKEN>"

# HuggingFace token — required for gated models; recommended for all models to avoid rate limits
HF_TOKEN = "<REPLACE WITH HF TOKEN>"

# ============================================================================
# KUBERNETES CLIENT CONFIGURATION
# ============================================================================
# The Configuration object holds the API server URL, auth token, and TLS settings.
# This is passed to the ApiClient, which the TrainerClient uses for all cluster operations.
configuration = client.Configuration()
configuration.host = api_server

# Uncomment if your cluster API server uses a self-signed certificate or an untrusted CA
# configuration.verify_ssl = False

configuration.api_key = {"authorization": f"Bearer {token}"}
api_client = client.ApiClient(configuration)

# ============================================================================
# PVC CONFIGURATION
# ============================================================================
# PVC_NAME must match the ReadWriteMany (RWX) PVC attached to your workbench.
# The notebook sees it at /opt/app-root/src/<pvc-name> (OpenShift AI convention).
# Training pods see it at /mnt/kubeflow-checkpoints (SDK constant CHECKPOINT_MOUNT_PATH).
PVC_NAME = "shared"
NOTEBOOK_SHARED_PATH = f"/opt/app-root/src/{PVC_NAME}"
SDK_MOUNT_PATH = "/mnt/kubeflow-checkpoints"

# Quick sanity check to help users discover the right workbench mount
if not os.path.exists(NOTEBOOK_SHARED_PATH):
    print(
        f"Warning: Expected workbench PVC mount not found at: {NOTEBOOK_SHARED_PATH}\n"
        "If your PVC has a different name or mount, update PVC_NAME above.\n"
        "Tip: in a workbench, PVCs are typically under /opt/app-root/src/."
    )

# ============================================================================
# MODEL CONFIGURATION
# ============================================================================
MODEL_NAME = "Qwen/Qwen3-8B"

# Create the TrainerClient — the main SDK entry point for submitting and managing TrainJobs.
# It uses the Kubernetes client configuration to authenticate against the cluster API.
trainer_client = TrainerClient(
    backend_config=KubernetesBackendConfig(
        client_configuration=api_client.configuration
    )
)

# ClusterTrainingRuntime (CTR) names — each speculator mode requires a specific CTR
# that defines the pod template, container images, and resource policies.
VLLM_EXTRACT_CTR = "vllm-extract-cuda"              # Used by DATA_ONLY and ONLINE: deploys vLLM sidecar
MODEL_OPT_CTR = "speculator-model-opt-cuda"         # Used by TRAIN_ONLY and OFFLINE: training only


print(f"API Server: {api_server}")
print(f"Model: {MODEL_NAME}")
print(f"PVC name: {PVC_NAME}")
print(f"Workbench PVC mount: {NOTEBOOK_SHARED_PATH}")
print(f"Training pod PVC mount (SDK): {SDK_MOUNT_PATH}")

## Shared Configuration

The following constants are shared across all examples. The verifier model is
[Qwen/Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B), a 36-layer transformer,
pre-downloaded to the shared PVC. All storage paths use **PVC URIs**
(`pvc://<pvc-name>/<path>`), which the SDK resolves to container mount paths internally.

When the verifier model is a PVC URI, the SDK cannot read the model config to
auto-detect layer IDs, so `target_layer_ids` must be provided explicitly via
`SpeculatorConfig`.

`RUN_NAME` creates a namespace on the PVC for each experiment — change it to start
a fresh run without overwriting previous results.

In [ ]:
# Unique run identifier — namespaces all output paths on the PVC.
# Change this to start a fresh experiment without overwriting previous results.
RUN_NAME = "run-01"

# PVC URI for the verifier model (pre-downloaded in the previous step).
# When using a PVC URI (vs. a HuggingFace model ID), the SDK cannot read the model
# config to auto-detect layer IDs, so TARGET_LAYER_IDS must be set manually.
VERIFIER_MODEL = MODEL_NAME

# Eagle3 reads hidden states from 4 intermediate layers of the verifier.
# Qwen3-8B has 36 transformer layers (indexed 1-36).
# Layers chosen: early (3), mid (18), late (33), and final (36) — giving the
# draft model a spread of low-level, mid-level, and high-level representations.
TARGET_LAYER_IDS = [3, 18, 33, 36]

# GPU, CPU, and memory allocations for the training container.
# 2 GPUs enable data-parallel training; 64Gi memory holds model weights + optimizer state.
TRAINING_RESOURCES = {
    "nvidia.com/gpu": 2,
    "cpu": "4",
    "memory": "64Gi",
}

# Resources for the vLLM sidecar that serves the verifier during extraction.
# 1 GPU is sufficient since vLLM only runs inference (no training).
# 64Gi memory is needed because vLLM loads the full model weights into CPU/GPU memory.
VLLM_RESOURCES = {
    "nvidia.com/gpu": 1,
    "cpu": "2",
    "memory": "64Gi",
}

# Training hyperparameters
EPOCHS = 3              # Number of full passes over the training data
LEARNING_RATE = 1e-4    # AdamW learning rate — 1e-4 is a good starting point for Eagle3
TOTAL_SEQ_LEN = 2048    # Maximum sequence length for both extraction and training
MAX_SAMPLES = 500       # Cap on the number of dataset samples to process

print("Shared Configuration:")
print(f"  Run name:          {RUN_NAME}")
print(f"  Verifier model:    {VERIFIER_MODEL}")
print(f"  Target layers:     {TARGET_LAYER_IDS}")
print(f"  Training GPUs:     {TRAINING_RESOURCES['nvidia.com/gpu']}")
print(f"  vLLM GPUs:         {VLLM_RESOURCES['nvidia.com/gpu']}")
print(f"  Epochs:            {EPOCHS}")
print(f"  Learning rate:     {LEARNING_RATE}")
print(f"  Sequence length:   {TOTAL_SEQ_LEN}")
print(f"  Max samples:       {MAX_SAMPLES}")

## 1. Data Extraction (DATA_ONLY Mode)

The `DATA_ONLY` mode extracts hidden states from the verifier model without performing
any training. The SDK deploys a managed vLLM sidecar alongside the job pod to serve the
verifier model. The sidecar processes the dataset and writes hidden state tensors
(`.safetensors` files) to the output PVC.

This is useful when you want to:
- Separate data extraction from training (extract once, experiment many times)
- Share extracted data across multiple training runs with different hyperparameters

**Key parameters explained:**
- `speculator_type` — Explicitly selects the Eagle3 architecture for the draft model
- `vllm_gpu_memory_utilization` — Fraction of GPU memory the vLLM sidecar can use (0.9 = 90%)
- `regenerate_responses` — When `True`, the vLLM sidecar generates new responses from the
  dataset prompts before extracting hidden states, rather than using the dataset's existing responses
- `datagen_concurrency` — Number of concurrent data generation workers
- `hidden_states_dtype` — Data type for saved hidden state tensors (`bfloat16` saves disk space)

**Not needed**: `training_resources` (no training happens in this mode)

We use the `ultrachat` built-in dataset for this example.

In [ ]:
# Job name follows a consistent naming convention: <architecture>-<mode>-<run>
DATA_JOB = f"eagle3-data-{RUN_NAME}"

# Output directory on PVC — hidden states will be written to <output_dir>/hidden_states/
DATA_ONLY_OUTPUT = f"pvc://{PVC_NAME}/speculator/{RUN_NAME}"

# Configure the DATA_ONLY trainer.
# This mode ONLY extracts hidden states — no training happens.
# The SDK deploys a managed vLLM sidecar to serve the verifier model.
data_only_trainer = SpeculativeDecodingTrainer(
    mode=SpeculatorMode.DATA_ONLY,                                      # Extract hidden states only, no training
    speculator_type=SpeculatorType.EAGLE3,                              # Use Eagle3 draft model architecture
    verifier_model=VERIFIER_MODEL,                                      # PVC path to the pre-downloaded verifier
    dataset_name="ultrachat",                                           # Built-in multi-turn conversational dataset
    max_samples=MAX_SAMPLES,                                            # Limit to 500 samples for this demo
    total_seq_len=TOTAL_SEQ_LEN,                                        # Truncate/pad sequences to 2048 tokens
    vllm_resources=VLLM_RESOURCES,                                      # GPU/CPU/memory for the vLLM sidecar
    vllm_gpu_memory_utilization=0.9,                                    # Let vLLM use 90% of GPU memory for KV cache
    regenerate_responses=True,                                          # Generate fresh responses from prompts (not reuse dataset answers)
    enable_progression_tracking=True,                                   # Enable SDK-side progress polling
    packages_to_install=["speculators==0.6.0","torchvision==0.24.0"],
    output_dir=DATA_ONLY_OUTPUT,                                        # PVC path where hidden states are saved
    config=SpeculatorConfig(
        target_layer_ids=TARGET_LAYER_IDS,                              # Which verifier layers to extract hidden states from
        datagen_concurrency=4,                                          # Number of parallel data generation workers
        hidden_states_dtype="bfloat16",                                 # Save tensors in bfloat16 to halve disk usage
    ),
    env={"HF_TOKEN": HF_TOKEN},                                         # Pass HF token to pods for gated model access
)

print("DATA_ONLY Configuration:")
print(f"  Job name:      {DATA_JOB}")
print(f"  Mode:          {data_only_trainer.mode.value}")
print(f"  Verifier:      {data_only_trainer.verifier_model}")
print(f"  Dataset:       {data_only_trainer.dataset_name}")
print(f"  Max samples:   {data_only_trainer.max_samples}")
print(f"  Target layers: {data_only_trainer.config.target_layer_ids}")
print(f"  Output dir:    {data_only_trainer.output_dir}")

In [ ]:
# Submit the DATA_ONLY TrainJob to the cluster.
# Name(...) assigns an explicit job name (otherwise the SDK auto-generates one).
# The runtime selects the CTR that includes a vLLM sidecar for hidden state extraction.
trainer_client.train(
    options=[Name(name=DATA_JOB)],
    trainer=data_only_trainer,
    runtime=DATA_EXTRACT_CTR,
)

print(f"DATA_ONLY job submitted: {DATA_JOB}")
print(f"\nMonitor logs with:")
print(f"  oc logs -f -l batch.kubernetes.io/job-name={DATA_JOB}-node-0 -c node")

In [ ]:
# Check the current status of the DATA_ONLY job (Pending, Running, Succeeded, Failed).
# Re-run this cell periodically to poll for completion.
trainer_client.get_job(DATA_JOB)

## 2. Train from Extracted Data (TRAIN_ONLY Mode)

The `TRAIN_ONLY` mode trains the Eagle3 draft model using hidden states extracted in
a previous `DATA_ONLY` run. No vLLM sidecar is needed — the training container reads
directly from the PVC.

Only the draft model's small components are trained:
- **FC layer 1 (fusion)**: Combines hidden states from four verifier layers into one vector
- **FC layer 2 (concat)**: Merges the fused hidden state with the previous token's embedding
- **One Transformer decoder layer**: Predicts the next token probability distribution

The verifier model is frozen — never modified.

**Key parameters explained:**
- `hidden_states_path` — Points to the `hidden_states/` subdirectory created by `DATA_ONLY`
- `data_path` — Points to the `DATA_ONLY` output directory (contains the preprocessed dataset)
- `num_layers` — Number of Transformer decoder layers in the draft model (1 is default)
- `ttt_steps` — Test-time training steps per batch
- `norm_before_residual` — Apply layer normalization before the residual connection
- `scheduler_type` — Learning rate scheduler (`"linear"` decays LR linearly to zero)
- `checkpoint_freq` — Save a checkpoint every N epochs (1.0 = every epoch)
- `resume_from_checkpoint` — If `True`, resumes training from the latest checkpoint if one exists

> **Note:** This example requires a completed `DATA_ONLY` run (Example 1 above).
> The `hidden_states_path` and `data_path` reference the output from that run.

In [ ]:
TRAIN_JOB = f"eagle3-train-{RUN_NAME}"
TRAIN_ONLY_OUTPUT = f"pvc://{PVC_NAME}/speculator/{RUN_NAME}/checkpoints"

# Configure the TRAIN_ONLY trainer.
# This mode trains the Eagle3 draft model from pre-extracted hidden states.
# No vLLM sidecar is deployed — the training container reads directly from PVC.
train_only_trainer = SpeculativeDecodingTrainer(
    mode=SpeculatorMode.TRAIN_ONLY,                         # Train from existing hidden states
    speculator_type=SpeculatorType.EAGLE3,
    verifier_model=VERIFIER_MODEL,
    hidden_states_path=f"{DATA_ONLY_OUTPUT}/hidden_states",  # Points to DATA_ONLY output
    data_path=DATA_ONLY_OUTPUT,                              # Preprocessed dataset from DATA_ONLY
    training_resources=TRAINING_RESOURCES,                   # 2 GPUs for data-parallel training
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    total_seq_len=TOTAL_SEQ_LEN,
    output_dir=TRAIN_ONLY_OUTPUT,
    config=SpeculatorConfig(
        target_layer_ids=TARGET_LAYER_IDS,
        num_layers=1,                  # Number of transformer decoder layers in the draft model
        ttt_steps=3,                   # Test-time training steps per batch (refine on each batch)
        norm_before_residual=True,     # Apply LayerNorm before adding the residual connection
        scheduler_type="linear",       # Linearly decay learning rate to zero over training
        checkpoint_freq=1.0,           # Save a checkpoint after every epoch
        resume_from_checkpoint=True,   # Resume from the latest checkpoint if one exists
    ),
    env={"HF_TOKEN": HF_TOKEN},
    enable_progression_tracking=False,
)

print("TRAIN_ONLY Configuration:")
print(f"  Job name:           {TRAIN_JOB}")
print(f"  Mode:               {train_only_trainer.mode.value}")
print(f"  Verifier:           {train_only_trainer.verifier_model}")
print(f"  Hidden states path: {train_only_trainer.hidden_states_path}")
print(f"  Data path:          {train_only_trainer.data_path}")
print(f"  Target layers:      {train_only_trainer.config.target_layer_ids}")
print(f"  Epochs:             {train_only_trainer.epochs}")
print(f"  Learning rate:      {train_only_trainer.lr}")
print(f"  Output dir:         {train_only_trainer.output_dir}")

In [ ]:
# Submit the TRAIN_ONLY TrainJob to the cluster.
# Uses MODEL_OPT_CTR since no vLLM sidecar is needed — training only.
trainer_client.train(
    options=[Name(name=TRAIN_JOB)],
    trainer=train_only_trainer,
    runtime=MODEL_OPT_CTR,
)

print(f"TRAIN_ONLY job submitted: {TRAIN_JOB}")
print(f"\nMonitor logs with:")
print(f"  oc logs -f -l batch.kubernetes.io/job-name={TRAIN_JOB}-node-0 -c node")

In [ ]:
# Check the current status of the TRAIN_ONLY job.
# Re-run this cell periodically to poll for completion.
trainer_client.get_job(TRAIN_JOB)

## 3. Offline Mode (User-Managed vLLM)

The `OFFLINE` mode extracts hidden states via a user-managed external vLLM server,
then trains the draft model in a single job. This is useful when you already have a
vLLM deployment running (e.g., as an OpenShift AI model serving instance) and want to
reuse it for hidden state extraction instead of having the SDK deploy a sidecar.

**How it works:**
1. The job connects to your external vLLM endpoint to extract hidden states
2. Hidden states are saved to the PVC at `hidden_states_path`
3. Training runs immediately after extraction completes — all within the same job

**Key differences from other modes:**
- You must provide `vllm_endpoint` pointing to your external vLLM server
- The SDK does not deploy a vLLM sidecar — `vllm_resources` is not used
- Both `training_resources` (for the training container) and `vllm_endpoint`
  (for extraction) are required

We use the `magpie` built-in dataset for this example.

In [ ]:
# URL of your externally managed vLLM server.
# This must be a running vLLM instance serving the same verifier model (Qwen3-8B) 
# and it also has to use the same target_layer_ids like the training job.
# The /v1 path exposes the OpenAI-compatible API that the SDK calls for extraction.
VLLM_ENDPOINT = "http://vllm-svc.speculative-decoding.svc.cluster.local:8000/v1"

OFFLINE_JOB = f"eagle3-offline-{RUN_NAME}"
OFFLINE_OUTPUT = f"pvc://{PVC_NAME}/speculator/{RUN_NAME}-offline"

# Configure the OFFLINE trainer.
# OFFLINE mode connects to an external vLLM endpoint for extraction, then trains.
# Both steps happen within the same job — extraction first, training second.
# Unlike DATA_ONLY + TRAIN_ONLY, this is a single-job workflow.
# Unlike ONLINE, the SDK does NOT deploy a vLLM sidecar — you manage it yourself.
offline_trainer = SpeculativeDecodingTrainer(
    mode=SpeculatorMode.OFFLINE,
    speculator_type=SpeculatorType.EAGLE3,
    verifier_model=VERIFIER_MODEL,
    dataset_name="magpie",                                    # Built-in Magpie conversation dataset
    max_samples=MAX_SAMPLES,
    total_seq_len=TOTAL_SEQ_LEN,
    vllm_endpoint=VLLM_ENDPOINT,                              # External vLLM server URL
    hidden_states_path=f"{OFFLINE_OUTPUT}/hidden_states",      # Where extracted states are saved
    training_resources=TRAINING_RESOURCES,                     # Resources for the training container
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    output_dir=OFFLINE_OUTPUT,
    config=SpeculatorConfig(
        target_layer_ids=TARGET_LAYER_IDS,
    ),
    env={"HF_TOKEN": HF_TOKEN},
    enable_progression_tracking=False,
)

print("OFFLINE Configuration:")
print(f"  Job name:         {OFFLINE_JOB}")
print(f"  Mode:             {offline_trainer.mode.value}")
print(f"  Verifier (PVC):   {offline_trainer.verifier_model}")
print(f"  vLLM endpoint:    {offline_trainer.vllm_endpoint}")
print(f"  Dataset:          {offline_trainer.dataset_name}")
print(f"  Target layers:    {offline_trainer.config.target_layer_ids}")
print(f"  Hidden states:    {offline_trainer.hidden_states_path}")
print(f"  Output dir:       {offline_trainer.output_dir}")

In [ ]:
# Submit the OFFLINE TrainJob to the cluster.
# Uses MODEL_OPT_CTR — no SDK-managed vLLM sidecar (the external endpoint handles extraction).
trainer_client.train(
    options=[Name(name=OFFLINE_JOB)],
    trainer=offline_trainer,
    runtime=MODEL_OPT_CTR,
)

print(f"OFFLINE job submitted: {OFFLINE_JOB}")
print(f"\nMonitor logs with:")
print(f"  oc logs -f -l batch.kubernetes.io/job-name={OFFLINE_JOB}-node-0 -c node")

In [ ]:
# Check the current status of the OFFLINE job.
# Re-run this cell periodically to poll for completion.
trainer_client.get_job(OFFLINE_JOB)

## 4. Online Mode (End-to-End Managed)

The `ONLINE` mode is the simplest workflow — the SDK manages everything in a single job:

1. **Deploys a vLLM sidecar** to serve the verifier model
2. **Extracts hidden states** from the dataset batch by batch
3. **Trains the Eagle3 draft model** using the extracted hidden states

Hidden states are processed in a streaming fashion — each batch is extracted, used for
training, then discarded. This means disk usage stays constant regardless of dataset
size, unlike `DATA_ONLY` which saves all hidden states to disk.

**Trade-off:** ONLINE is simpler (one step instead of two), but you cannot reuse the
extracted data for multiple training runs with different hyperparameters. If you want to
experiment with hyperparameters, use `DATA_ONLY` + `TRAIN_ONLY` instead.

We use the `magpie` built-in dataset for this example.

In [ ]:
ONLINE_JOB = f"eagle3-online-{RUN_NAME}"
ONLINE_OUTPUT = f"pvc://{PVC_NAME}/speculator/{RUN_NAME}-online"

# Configure the ONLINE trainer — the simplest end-to-end workflow.
# The SDK manages everything: deploys vLLM sidecar, extracts hidden states, and trains.
# Hidden states are streamed batch-by-batch and discarded after use, so disk usage
# stays constant regardless of dataset size (unlike DATA_ONLY which persists all states).
online_trainer = SpeculativeDecodingTrainer(
    mode=SpeculatorMode.ONLINE,                # Fully managed: extraction + training in one step
    speculator_type=SpeculatorType.EAGLE3,
    verifier_model=VERIFIER_MODEL,
    dataset_name="magpie",
    max_samples=MAX_SAMPLES,
    total_seq_len=TOTAL_SEQ_LEN,
    vllm_resources=VLLM_RESOURCES,             # Resources for the managed vLLM sidecar
    vllm_gpu_memory_utilization=0.9,
    training_resources=TRAINING_RESOURCES,      # Resources for the training container
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    output_dir=ONLINE_OUTPUT,
    config=SpeculatorConfig(
        target_layer_ids=TARGET_LAYER_IDS,
    ),
    env={"HF_TOKEN": HF_TOKEN},
    enable_progression_tracking=False,
)

print("ONLINE Configuration:")
print(f"  Job name:      {ONLINE_JOB}")
print(f"  Mode:          {online_trainer.mode.value}")
print(f"  Verifier:      {online_trainer.verifier_model}")
print(f"  Dataset:       {online_trainer.dataset_name}")
print(f"  Max samples:   {online_trainer.max_samples}")
print(f"  Target layers: {online_trainer.config.target_layer_ids}")
print(f"  Training GPUs: {TRAINING_RESOURCES['nvidia.com/gpu']}")
print(f"  vLLM GPUs:     {VLLM_RESOURCES['nvidia.com/gpu']}")
print(f"  Output dir:    {online_trainer.output_dir}")

In [ ]:
# Submit the ONLINE TrainJob to the cluster.
# Uses ONLINE_CTR which includes both a vLLM sidecar and training container.
trainer_client.train(
    options=[Name(name=ONLINE_JOB)],
    trainer=online_trainer,
    runtime=ONLINE_CTR,
)

print(f"ONLINE job submitted: {ONLINE_JOB}")
print(f"\nMonitor logs with:")
print(f"  oc logs -f -l batch.kubernetes.io/job-name={ONLINE_JOB}-node-0 -c node")

In [ ]:
# Check the current status of the ONLINE job.
# Re-run this cell periodically to poll for completion.
trainer_client.get_job(ONLINE_JOB)

## 5. Cleanup

Delete the TrainJobs when you are done. Uncomment the jobs you submitted.

In [ ]:
# Delete completed TrainJobs to free cluster resources (pods, volumes, etc.).
# Uncomment only the jobs you actually submitted above.
# Note: Deleting a job does NOT delete the output data on the PVC —
# checkpoints and hidden states remain available for future use.

# trainer_client.delete_job(DATA_JOB)
# trainer_client.delete_job(TRAIN_JOB)
# trainer_client.delete_job(OFFLINE_JOB)
# trainer_client.delete_job(ONLINE_JOB)

# print("TrainJobs deleted.")

## Summary

This notebook demonstrated four ways to train an Eagle3 draft model for speculative decoding:

1. **DATA_ONLY** — Extracted hidden states from Qwen3-8B using a managed vLLM sidecar
   and the `ultrachat` dataset. Output stored on PVC for reuse.
2. **TRAIN_ONLY** — Trained the draft model from the pre-extracted hidden states
   produced by the DATA_ONLY run. No vLLM needed.
3. **OFFLINE** — Used an external vLLM endpoint with the `magpie` dataset.
   Extracted hidden states and trained in a single job.
4. **ONLINE** — Fully managed end-to-end training with the `magpie` dataset.
   The SDK handled vLLM deployment, data extraction, and training in one step.

### Key Takeaways

- All storage paths use **PVC URIs** (`pvc://<pvc-name>/<path>`) — the SDK handles
  volume mounting internally. S3 URIs are not yet supported for speculator training.
- Use `RUN_NAME` to namespace your experiments on the PVC — change it to start a fresh
  run without overwriting previous results.
- Use `options=[Name(name="...")]` to give jobs explicit names for easier monitoring
  and status checking.
- Pass `env={"HF_TOKEN": ...}` to each trainer so training pods can access gated models.
- When using a HuggingFace model ID, the SDK **auto-detects** `target_layer_ids`.
  When using a PVC URI, you must provide them explicitly via `SpeculatorConfig`.
- The DATA_ONLY + TRAIN_ONLY split lets you extract data once and iterate on training
  hyperparameters without re-running the expensive extraction step.
- For production workloads, ONLINE mode provides the simplest single-step experience.

### Next Steps

- Adjust `epochs`, `lr`, and `max_samples` to tune draft model quality.
- Use `SpeculatorConfig` to customize architecture parameters (e.g., `num_layers`,
  `ttt_steps`, `scheduler_type`).
- Set `resume_from_checkpoint=True` in `SpeculatorConfig` to resume training after
  interruptions.
- Deploy the trained draft model with vLLM for speculative decoding inference.